In [1]:
import pandas as pd
from src.training.bert_pipeline import TrainingBertPipeline
import logging
import torch
import os

In [2]:
df = pd.read_csv("data/aes_dataset_5k_clean.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5363 entries, 0 to 5362
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   question          4859 non-null   object 
 1   reference_answer  5363 non-null   object 
 2   answer            5363 non-null   object 
 3   score             5363 non-null   float64
 4   normalized_score  5363 non-null   float64
 5   multibert_length  5363 non-null   int64  
 6   dataset           5363 non-null   object 
 7   dataset_num       5363 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 335.3+ KB


In [3]:
df['dataset'].value_counts()

dataset
sag               2558
analisis_essay    2162
stita              333
cunlp              171
sci                139
Name: count, dtype: int64

In [4]:
# Check if the first file exists
df_result = None
if os.path.exists("experiments/results/results.csv"):
    df_result = pd.read_csv("experiments/results/results.csv")
    print(df_result['config_id'].iloc[-1])
else:
    print("File 'results.csv' does not exist.")

83


In [5]:
batch_sizes = [4, 8]
overlappings = [64]
epochs_list = [5, 10]
learning_rates = [1e-5, 2e-5, 5e-5]
idx = (df_result['config_id'].iloc[-1] + 1) if df_result is not None and not df_result.empty else 0  # index untuk setiap kombinasi
ROOT_DIR = os.getcwd()

In [6]:
# model = [
#     ("bert_length", "bert-base-uncased"),
#     ("indobert_length", "indobenchmark/indobert-base-p1"),
#     ("albert_length", "albert-base-v1"),
#     ("indoalbert_length", "indobenchmark/indobert-lite-base-p2"),
#     ("longformer_length", "allenai/longformer-base-4096"),
#     ("multibert_length", "google-bert/bert-base-multilingual-uncased")
# ]

In [7]:
for batch_size in batch_sizes:
    for overlapping in overlappings:
        for num_epochs in epochs_list:
            for lr in learning_rates:
                results = []
                results_epoch = []
                df_result1 = None
                # Check if the second file exists
                if os.path.exists("experiments/results/results_epoch.csv"):
                    df_result1 = pd.read_csv("experiments/results/results_epoch.csv")
                    print(max(df_result1['valid_qwk']))
                else:
                    print("File 'results_epoch.csv' does not exist.")
                config = {
                    "df": df,
                    "model_name": "google-bert/bert-base-multilingual-uncased",
                    "overlapping": overlapping,
                    "batch_size": batch_size,
                    "learning_rate": lr,
                    "epochs": num_epochs,
                    "config_id": idx,
                    "max_seq_len": 128,
                    "col_length": "multibert_length",
                    "best_valid_qwk": max(df_result1['valid_qwk']) if df_result1 is not None and not df_result1.empty else float("-inf")
                }

                logging.info(
                    f"Running configuration: config_id={idx}, model_name={config['model_name']}, batch_size={batch_size}, "
                    f"max_seq_length={config['max_seq_len']}, overlapping={overlapping}, epochs={num_epochs}, learning_rate={lr}"
                )
                
                print(
                    f"\nRunning configuration: config_id={idx}, model_name={config['model_name']}, batch_size={batch_size}, "
                    f"max_seq_length={config['max_seq_len']}, overlapping={overlapping}, epochs={num_epochs}, learning_rate={lr}"
                )
                
                try:
                    pipeline = TrainingBertPipeline(config, results, results_epoch)
                    pipeline.run_training()

                    # Save results
                    # Dapatkan root project
                    results_path = os.path.join(ROOT_DIR, "experiments/results/results.csv")
                    results_epoch_path = os.path.join(ROOT_DIR, "experiments/results/results_epoch.csv")
                    TrainingBertPipeline.save_csv(results, results_path)
                    TrainingBertPipeline.save_csv(results_epoch, results_epoch_path)
                except Exception as e:
                    logging.error(f"Error in config_id={idx}: {str(e)}")
                    print(f"Error in config_id={idx}: {str(e)}")
                    torch.cuda.empty_cache()
                finally:
                    # Clear GPU memory after every configuration
                    del pipeline.model
                    del pipeline.tokenizer
                    del pipeline.optimizer
                    torch.cuda.empty_cache()

                idx += 1

0.9251343231432688

Running configuration: config_id=84, model_name=google-bert/bert-base-multilingual-uncased, batch_size=4, max_seq_length=128, overlapping=64, epochs=5, learning_rate=1e-05
split dataset run...
create dataset run...
max len 128
max len 128
max len 128
create dataloader run...
====== Training Epoch 1/5 ======


Token indices sequence length is longer than the specified maximum sequence length for this model (1004 > 512). Running this sequence through the model will result in indexing errors
c:\Users\User\Documents\Code\env\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Train Loss: 0.0506, Train QWK: 0.6041, Train Pearson: 0.7397
Validation Loss: 0.0309, Validation QWK: 0.8551, Validation Pearson: 0.8799
====== Training Epoch 2/5 ======
Train Loss: 0.0298, Train QWK: 0.7065, Train Pearson: 0.8519
Validation Loss: 0.0242, Validation QWK: 0.8879, Validation Pearson: 0.8935
====== Training Epoch 3/5 ======
Train Loss: 0.0224, Train QWK: 0.7269, Train Pearson: 0.8900
Validation Loss: 0.0224, Validation QWK: 0.9011, Validation Pearson: 0.9021
====== Training Epoch 4/5 ======
Train Loss: 0.0193, Train QWK: 0.7724, Train Pearson: 0.9062
Validation Loss: 0.0277, Validation QWK: 0.8754, Validation Pearson: 0.8905
====== Training Epoch 5/5 ======
Train Loss: 0.0149, Train QWK: 0.8001, Train Pearson: 0.9281
Validation Loss: 0.0444, Validation QWK: 0.8436, Validation Pearson: 0.8891
Test Loss: 0.0405, Test QWK: 0.8598, Test Pearson: 0.8974
0.9251343231432688

Running configuration: config_id=85, model_name=google-bert/bert-base-multilingual-uncased, batch_size=4,

Token indices sequence length is longer than the specified maximum sequence length for this model (1004 > 512). Running this sequence through the model will result in indexing errors


Train Loss: 0.0552, Train QWK: 0.5989, Train Pearson: 0.7060
Validation Loss: 0.0464, Validation QWK: 0.7891, Validation Pearson: 0.8490
====== Training Epoch 2/5 ======
Train Loss: 0.0337, Train QWK: 0.6837, Train Pearson: 0.8300
Validation Loss: 0.0468, Validation QWK: 0.7639, Validation Pearson: 0.8027
====== Training Epoch 3/5 ======
Train Loss: 0.0277, Train QWK: 0.7351, Train Pearson: 0.8625
Validation Loss: 0.0228, Validation QWK: 0.8880, Validation Pearson: 0.8939
====== Training Epoch 4/5 ======
Train Loss: 0.0210, Train QWK: 0.7658, Train Pearson: 0.8974
Validation Loss: 0.0281, Validation QWK: 0.8765, Validation Pearson: 0.8852
====== Training Epoch 5/5 ======
Train Loss: 0.0171, Train QWK: 0.8113, Train Pearson: 0.9175
Validation Loss: 0.0348, Validation QWK: 0.8317, Validation Pearson: 0.8924
Test Loss: 0.0311, Test QWK: 0.8514, Test Pearson: 0.9012
0.9251343231432688

Running configuration: config_id=86, model_name=google-bert/bert-base-multilingual-uncased, batch_size=4,

...

Train Loss: 0.0504, Train QWK: 0.6184, Train Pearson: 0.7344
Validation Loss: 0.0691, Validation QWK: 0.6956, Validation Pearson: 0.7757
====== Training Epoch 2/10 ======
Train Loss: 0.0377, Train QWK: 0.6754, Train Pearson: 0.8095
Validation Loss: 0.0242, Validation QWK: 0.8812, Validation Pearson: 0.8918
====== Training Epoch 3/10 ======
Train Loss: 0.0281, Train QWK: 0.7220, Train Pearson: 0.8602
Validation Loss: 0.0250, Validation QWK: 0.8813, Validation Pearson: 0.8939
====== Training Epoch 4/10 ======
Train Loss: 0.0235, Train QWK: 0.7578, Train Pearson: 0.8846
Validation Loss: 0.0244, Validation QWK: 0.8830, Validation Pearson: 0.8863
====== Training Epoch 5/10 ======
Train Loss: 0.0197, Train QWK: 0.7756, Train Pearson: 0.9042
Validation Loss: 0.0294, Validation QWK: 0.8816, Validation Pearson: 0.8905
====== Training Epoch 6/10 ======
Train Loss: 0.0165, Train QWK: 0.7903, Train Pearson: 0.9205
Validation Loss: 0.0256, Validation QWK: 0.8916, Validation Pearson: 0.8946
====== T

Token indices sequence length is longer than the specified maximum sequence length for this model (1004 > 512). Running this sequence through the model will result in indexing errors


Train Loss: 0.0487, Train QWK: 0.6342, Train Pearson: 0.7440
Validation Loss: 0.0671, Validation QWK: 0.7448, Validation Pearson: 0.8560
====== Training Epoch 2/10 ======
Train Loss: 0.0299, Train QWK: 0.7073, Train Pearson: 0.8510
Validation Loss: 0.0271, Validation QWK: 0.8778, Validation Pearson: 0.8883
====== Training Epoch 3/10 ======
Train Loss: 0.0227, Train QWK: 0.7394, Train Pearson: 0.8889
Validation Loss: 0.0245, Validation QWK: 0.8803, Validation Pearson: 0.8965
====== Training Epoch 4/10 ======
Train Loss: 0.0182, Train QWK: 0.7961, Train Pearson: 0.9123
Validation Loss: 0.0294, Validation QWK: 0.8866, Validation Pearson: 0.8830
====== Training Epoch 5/10 ======
Train Loss: 0.0155, Train QWK: 0.7999, Train Pearson: 0.9256
Validation Loss: 0.0253, Validation QWK: 0.8893, Validation Pearson: 0.8954
====== Training Epoch 6/10 ======
Train Loss: 0.0125, Train QWK: 0.8271, Train Pearson: 0.9403
Validation Loss: 0.0307, Validation QWK: 0.8826, Validation Pearson: 0.9027
====== T

Token indices sequence length is longer than the specified maximum sequence length for this model (1004 > 512). Running this sequence through the model will result in indexing errors


Train Loss: 0.0585, Train QWK: 0.6103, Train Pearson: 0.7003
Validation Loss: 0.0346, Validation QWK: 0.8559, Validation Pearson: 0.8694
====== Training Epoch 2/10 ======
Train Loss: 0.0322, Train QWK: 0.7035, Train Pearson: 0.8395
Validation Loss: 0.0212, Validation QWK: 0.8982, Validation Pearson: 0.9036
====== Training Epoch 3/10 ======
Train Loss: 0.0234, Train QWK: 0.7526, Train Pearson: 0.8855
Validation Loss: 0.0213, Validation QWK: 0.8992, Validation Pearson: 0.9010
====== Training Epoch 4/10 ======
Train Loss: 0.0190, Train QWK: 0.7809, Train Pearson: 0.9080
Validation Loss: 0.0273, Validation QWK: 0.8807, Validation Pearson: 0.8988
====== Training Epoch 5/10 ======
Train Loss: 0.0179, Train QWK: 0.7980, Train Pearson: 0.9133
Validation Loss: 0.0415, Validation QWK: 0.8252, Validation Pearson: 0.8826
====== Training Epoch 6/10 ======
Train Loss: 0.0153, Train QWK: 0.8015, Train Pearson: 0.9262
Validation Loss: 0.0317, Validation QWK: 0.8659, Validation Pearson: 0.8934
====== T